## 0. Uvod i importovanje

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("D:/Python/libraries/ecommerce_veliki_projekat_raw.csv")

## 1. Pregled i analiza  pocetnog stanja

In [2]:
print(df.info())
print(df.head(10).to_string()) # Bolji pregled sa to_string()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1255 entries, 0 to 1254
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Transakcija_ID  1240 non-null   object 
 1   Datum           1197 non-null   object 
 2   Kod_Proizvoda   1240 non-null   object 
 3   Cena_Tekst      1189 non-null   object 
 4   Količina        1240 non-null   float64
 5   Ocena_Kupca     1210 non-null   float64
 6   Godine_Kupca    1240 non-null   float64
 7   Grad            1218 non-null   object 
dtypes: float64(3), object(5)
memory usage: 78.6+ KB
None
  Transakcija_ID                Datum   Kod_Proizvoda Cena_Tekst  Količina  Ocena_Kupca  Godine_Kupca        Grad
0      TXN-10000  2025-12-18 11:00:00  CAT_BEAUTY-447     33.27$       2.0          4.6          47.0         Niš
1      TXN-10001  2025-03-05 23:00:00    CAT_HOME-466        NaN       2.0          3.7          67.0  Kragujevac
2      TXN-10002  2025-02-10 04:00:00  CAT_B

## 2. Ciscenje NaN sa subsetovima i thresh

In [3]:
print(df.isna().sum().to_string()) # Sve kolone imaju nepostojece vrednosti
df.dropna(axis=0, subset=["Datum", "Kod_Proizvoda"], how="any", inplace=True)
df.dropna(axis=0, thresh=4, inplace=True)
df.reset_index(drop=True, inplace=True)
print(df.isna().sum().to_string()) # Provera - Cnea_Tekst ima 50, Ocena_Kuca 29 i Grad 22 NaN vrednosti nakon ""dropna"


Transakcija_ID    15
Datum             58
Kod_Proizvoda     15
Cena_Tekst        66
Količina          15
Ocena_Kupca       45
Godine_Kupca      15
Grad              37
Transakcija_ID     0
Datum              0
Kod_Proizvoda      0
Cena_Tekst        50
Količina           0
Ocena_Kupca       29
Godine_Kupca       0
Grad              22


## 3. Ekstrakcija i ciscenje kolona


In [4]:
print(df["Cena_Tekst"].sample(10, random_state=42).to_string()) # Neki imaju znak dolara na pocetku ili kraju, sto izbacujemo

df["Cena"] = df["Cena_Tekst"].str.strip().str.replace("$", "", regex=False).astype("Float64").round(2)

print(df["Cena"].isna().sum()) # Imamo i dalje 50 komada nepostojecih svednosti
medijana_cena = df["Cena"].median()
df["Cena"] = df["Cena"].fillna(medijana_cena)

df.drop(columns=["Cena_Tekst"], inplace=True)

921       $286.93
321       319.25$
101         $31.4
920          74.7
58           70.5
790       498.86$
948        170.65
969        125.05
410        $425.8
1079     $302.64 
50


## 4. Parsiranje kolone "Kod_Proizvoda"

In [5]:
def parsiranje_koda(text):
    if pd.isna(text):
        return np.nan

    text = str(text).replace("CAT_", "")
    parts = str(text).strip().split("-")

    try:
        return parts[0]
    except(ValueError, IndexError):
        return np.nan

df["Kategorija"] = df["Kod_Proizvoda"].apply(parsiranje_koda).astype("category")
df.drop(columns=["Kod_Proizvoda"], inplace=True)

kategorija_mapping = {
    "BEAUTY": "Beauty",
    "BOOKS": "Books",
    "CLOTH": "Cloth",
    "HOME": "Home",
    "ELEC": "Electronics"
}

df["Kategorija"] = df["Kategorija"].astype(str).replace(kategorija_mapping).astype("category")
print(df["Kategorija"])

0            Beauty
1              Home
2            Beauty
3              Home
4       Electronics
           ...     
1192          Cloth
1193           Home
1194           Home
1195           Home
1196          Cloth
Name: Kategorija, Length: 1197, dtype: category
Categories (5, object): ['Beauty', 'Books', 'Cloth', 'Electronics', 'Home']


## 5. Ciscenje datuma i ekstrakcija vremenskih obelezja

In [6]:
df["Datum"] = pd.to_datetime(df["Datum"], format="mixed", errors="coerce")
df.dropna(axis=0, subset=["Datum"], how="any", inplace=True)

df["Mesec"] = df["Datum"].dt.month
df["Dan_u_nedelji"] = df["Datum"].dt.day_of_week
df["Sat"] = df["Datum"].dt.hour

# Ne znam bolji nacin za pravljenje naziva dana u nedelji pa cemo ovako, taman vezbamo mapping
day_of_the_week = {
    "0": "Monday",
    "1": "Tuesday",
    "2": "Wednesday",
    "3": "Thursday",
    "4": "Friday",
    "5": "Saturday",
    "6": "Sunday"
}

df["Dan_u_nedelji"] = df["Dan_u_nedelji"].astype(str).replace(day_of_the_week).astype("category")

## 6. Napredno uklanjanje duplikate

In [7]:
print(df.duplicated(subset=["Transakcija_ID"]).sum()) # Ima 36 duplikata

df.drop_duplicates(keep="first", subset=["Transakcija_ID"], inplace=True)
df.reset_index(drop=True, inplace=True)

36


## 7. Kreiranje izvedenih kolona i finalna optimizacija

In [8]:
df["Ukupan_Prihod"] = (df["Cena"] * df["Količina"]).round(2)

df["Grad"] = df["Grad"].astype("category")
df.dtypes # Sacu ja da menjam svoje

def parse_id(text):
    if pd.isna(text):
        return np.nan

    parts = str(text).strip().split("-")

    try:
        return int(parts[1])
    except(ValueError, IndexError):
        return np.nan


df["Transakcija_ID"] = df["Transakcija_ID"].apply(parse_id).astype("Int16")
df["Količina"] = df["Količina"].astype("Int8")
# df["Ocena_Kupca"] = df["Ocena_Kupca"].astype("Int8") ne mozemo zbog NaN vrednosti koje mi nije receno da izbacim
df["Godine_Kupca"] = df["Godine_Kupca"].astype("Int8")
df["Mesec"] = df["Mesec"].astype("Int8")
df["Sat"] = df["Sat"].astype("Int8")
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1096 entries, 0 to 1095
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Transakcija_ID  1096 non-null   Int16         
 1   Datum           1096 non-null   datetime64[ns]
 2   Količina        1096 non-null   Int8          
 3   Ocena_Kupca     1069 non-null   float64       
 4   Godine_Kupca    1096 non-null   Int8          
 5   Grad            1077 non-null   category      
 6   Cena            1096 non-null   Float64       
 7   Kategorija      1096 non-null   category      
 8   Mesec           1096 non-null   Int8          
 9   Dan_u_nedelji   1096 non-null   category      
 10  Sat             1096 non-null   Int8          
 11  Ukupan_Prihod   1096 non-null   Float64       
dtypes: Float64(2), Int16(1), Int8(4), category(3), datetime64[ns](1), float64(1)
memory usage: 52.3 KB
None


# Vizualizacija

In [9]:
import plotly.express as px

## 1. Barplot - Ukupan prihod po kategorijama

In [10]:
df_bar = df.groupby("Kategorija", observed=True).agg(prihod = ("Ukupan_Prihod", "sum")).reset_index()

fig = px.bar(df_bar,
             x = "Kategorija",
             y = "prihod",
             color = "Kategorija",
             template="plotly_dark")

fig.update_layout(title=dict(text="<b>Ukupan prihod po kategorijama</b>", font=dict(size=20), xanchor="center", x=0.5), 
                  paper_bgcolor = "#1e1e2e",
                  plot_bgcolor="#1e1e2e",
                  showlegend=False)

fig.update_traces(hovertemplate="<b>Kategorija:</b> %{x}<br><b>Ukupan Prihod:</b> $%{y:,.2f}<extra></extra>",
                  marker_line_color="#00f2fe",
                  marker_line_width=1.5)

fig.update_xaxes(showgrid=False, # Ovo je podrazumevano
                 title_font=dict(size=16)) 

fig.update_yaxes(gridcolor="#313244",
                 title="Ukupan Prihod",
                 title_font=dict(size=16))

fig.show()

## 2. Lineplot - Mesecni trend prodaje

In [11]:
df_line = df.groupby("Mesec", observed=False).agg(prihod=("Ukupan_Prihod", "sum")).reset_index()

fig2 = px.line(df_line,
               x = "Mesec",
               y = "prihod",
               markers=True,
               template="plotly_dark")

fig2.update_layout(title=dict(text="Mesecni trend prodaje", font=dict(size=20), xanchor="center", x=0.5),
                   paper_bgcolor="#1e1e2e",
                   plot_bgcolor="#1e1e2e")

fig2.update_traces(hovertemplate="<b>Mesec:</b> %{x}<br><b>Ukupan prihod:</b> $%{y:,.2f}<extra></extra>",
                   line_color="#ff007f", 
                   marker_size=8,
                   marker_color = "cyan",
                   marker_line_color= "white",
                   marker_line_width = 2)

fig2.update_xaxes(dtick=1,
                  title_font=dict(size=16))

fig2.update_yaxes(title="Prihod", 
                  title_font=dict(size=16),
                  tickformat=",.0f")

fig2.show()

## 3. Boxplot - Distribucija cena po kategorijama

In [12]:
fig3 = px.box(df,
               x = "Cena",
               y = "Kategorija",
               color = "Kategorija",
               color_discrete_map={"Books":"#FF0000", "Cloth":"#FF8400", "Electronics":"#E5FF00", "Home":"#51FF00", "Beauty":"#00F2FF"},
               template="plotly_dark")

fig3.update_layout(title=dict(text="Distribucija cena po kategorijama", font=dict(size=20), xanchor="center", x=0.5),
                   paper_bgcolor="#1e1e2e",
                   plot_bgcolor="#1e1e2e",
                   height=600)

fig3.update_traces(hovertemplate="<b>Cena:</b> $%{x:,.2f}<extra>%{y}</extra>", jitter=0.3, boxpoints="all")

fig3.update_xaxes(title_font=dict(size=16))

fig3.update_yaxes(title="Kategorija", 
                  title_font=dict(size=16),
                  tickformat=",.0f")

fig3.show()

## 4. Histograms - Distribucija godina kupaca sa marginal Box-om

In [13]:
fig4 = px.histogram(df,
                    x = "Godine_Kupca",
                    marginal="box",
                    nbins=20, # Svaki stub je 4 godine
                    template="plotly_dark")

fig4.update_layout(bargap=0.2,
                   paper_bgcolor="#1e1e2e",
                   plot_bgcolor="#1e1e2e",
                   title=dict(text="Distribucija godina kupaca", font=dict(size=20, color="cyan"), xanchor="center", x=0.5))

fig4.update_yaxes(title_text="", selector=dict(anchor="x"))  # Uklanja "count" / naziv sa Y2 ose
fig4.update_xaxes(title_text="", selector=dict(anchor="y2"))

fig4.update_xaxes(title="Godine Kupca", title_font=dict(size=16, color="cyan"), selector=dict(anchor="y"), dtick=10)
fig4.update_yaxes(title="Broj", title_font=dict(size=16, color="cyan"), selector=dict(anchor="x"))

fig4.update_traces(hovertemplate="<b>Godine:</b> %{x}<br><b>Broj:</b> %{y}",
                   marker_color = "#0073FF",
                   marker_line_color = "#41E6FF",
                   marker_line_width = 2)

fig4.show()

## 5. Scatter - Ocene kupca vs Ukupan prihod

In [14]:
fig5 = px.scatter(df,
                    x = "Ocena_Kupca",
                    y = "Ukupan_Prihod",
                    color = "Grad",
                    template="plotly_dark")

fig5.update_layout(title=dict(text="Ocene kupca vs Ukupan prihod", font=dict(size=20, color="cyan"), xanchor="center", x=0.5),
                   legend=dict(orientation="h", xanchor="center", x=0.5, yanchor="bottom", y=0.98),
                   paper_bgcolor="#1e1e2e",
                   plot_bgcolor="#1e1e2e",)

fig5.update_xaxes(title="Ocena Kupca", title_font=dict(size=16, color="cyan"))
fig5.update_yaxes(title="Ukupan Prihod", title_font=dict(size=16, color="cyan"))

fig5.update_traces(hovertemplate="<b>Ocena:</b> %{x}<br><b>Prihod:</b> %{y}<br><b>Grad:</b> %{fullData.name}<extra></extra>")

fig5.show()